# 03 — Churn Model Analysis
**XGBoost + SMOTE + Balanced Threshold (v3)**

Fixed: data leakage (recency_days removed), added behavioral features, balanced threshold.

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')
print("Setup complete")

import joblib
from config import MODELS_DIR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE


## 1. Load features (v3 — no recency_days)

In [ ]:

sql = '''SELECT f.customer_id,
COUNT(DISTINCT f.order_id) AS frequency,
ROUND(SUM(f.revenue)::numeric,2) AS monetary,
ROUND(AVG(f.price)::numeric,2) AS avg_order_value,
ROUND(AVG(f.review_score)::numeric,2) AS avg_review_score,
ROUND(AVG(f.freight_value/NULLIF(f.revenue,0))::numeric,4) AS avg_freight_pct,
ROUND(AVG(f.payment_installments)::numeric,2) AS avg_installments,
SUM(CASE WHEN f.delivery_delay_days>0 THEN 1 ELSE 0 END) AS late_deliveries,
ROUND(AVG(f.delivery_delay_days)::numeric,1) AS avg_delay_days,
COUNT(DISTINCT f.category) AS unique_categories,
COUNT(DISTINCT f.product_id) AS unique_products,
ROUND(MAX(f.price)::numeric,2) AS max_order_value,
ROUND(MIN(f.price)::numeric,2) AS min_order_value,
SUM(CASE WHEN f.review_score>=4 THEN 1 ELSE 0 END) AS positive_reviews,
SUM(CASE WHEN f.review_score<=2 THEN 1 ELSE 0 END) AS negative_reviews,
COUNT(DISTINCT f.order_year) AS active_years,
COUNT(DISTINCT f.order_month) AS active_months,
MAX(f.order_date)::date - MIN(f.order_date)::date AS customer_lifetime_days,
CASE WHEN COUNT(DISTINCT f.order_id)>1 THEN (MAX(f.order_date)::date-MIN(f.order_date)::date)/(COUNT(DISTINCT f.order_id)-1) ELSE 0 END AS avg_days_between_orders,
COUNT(DISTINCT f.category) AS category_diversity,
ROUND((SUM(f.revenue)/NULLIF(COUNT(DISTINCT f.order_id),0))::numeric,2) AS revenue_per_order,
DATE '2018-09-01' - MAX(f.order_date)::date AS recency_days
FROM olist.fact_orders f WHERE f.is_delivered=1 GROUP BY f.customer_id HAVING COUNT(DISTINCT f.order_id)>=1'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
df['churned'] = (df['recency_days']>180).astype(int)
df = df.drop(columns=['recency_days'])
features = ['frequency','monetary','avg_order_value','avg_review_score','avg_freight_pct',
            'avg_installments','late_deliveries','avg_delay_days','unique_categories',
            'unique_products','max_order_value','min_order_value','positive_reviews',
            'negative_reviews','active_years','active_months','customer_lifetime_days',
            'avg_days_between_orders','category_diversity','revenue_per_order']
X = df[features].fillna(0); y = df['churned']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler = joblib.load(os.path.join(MODELS_DIR,'churn_scaler.pkl'))
model  = joblib.load(os.path.join(MODELS_DIR,'churn_model.pkl'))
X_test_sc = scaler.transform(X_test)
y_pred = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:,1]
print("Model and data loaded")
print(classification_report(y_test,y_pred,target_names=['Active','Churned']))


## 2. Confusion matrix and ROC curve

In [ ]:

fig,axes = plt.subplots(1,2,figsize=(13,5))
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred),display_labels=['Active','Churned']).plot(ax=axes[0],cmap='Blues',colorbar=False)
axes[0].set_title('Confusion Matrix — Churn Model v3')
fpr,tpr,_ = roc_curve(y_test,y_proba)
roc_auc_val = auc(fpr,tpr)
axes[1].plot(fpr,tpr,color='steelblue',lw=2,label=f'ROC AUC = {roc_auc_val:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Churn Model v3 (leakage fixed)'); axes[1].legend()
plt.tight_layout(); plt.show()


## 3. Feature importance

In [ ]:

importance = pd.DataFrame({'feature':features,'importance':model.feature_importances_}).sort_values('importance',ascending=True)
plt.figure(figsize=(10,7))
plt.barh(importance['feature'],importance['importance'],color='steelblue')
plt.title('XGBoost Feature Importance — Churn Model v3'); plt.xlabel('Importance')
plt.tight_layout(); plt.show()
print("Top 5 features:")
for _,row in importance.tail(5)[::-1].iterrows():
    print(f"  {row['feature']:<30} {row['importance']:.4f}")


## 4. Balanced threshold analysis

In [ ]:

from sklearn.metrics import recall_score
thresholds = np.arange(0.2,0.8,0.01)
active_recalls, churned_recalls = [],[]
for t in thresholds:
    p = (y_proba>=t).astype(int)
    active_recalls.append(recall_score(y_test,p,pos_label=0))
    churned_recalls.append(recall_score(y_test,p,pos_label=1))
plt.figure(figsize=(10,5))
plt.plot(thresholds,active_recalls,label='Active recall',color='steelblue',lw=2)
plt.plot(thresholds,churned_recalls,label='Churned recall',color='darkorange',lw=2)
plt.axvline(0.53,color='red',linestyle='--',lw=1,label='Selected threshold 0.53')
plt.xlabel('Threshold'); plt.ylabel('Recall')
plt.title('Recall vs Threshold (balanced at 0.53)'); plt.legend()
plt.tight_layout(); plt.show()
